# Explainable & Calibrated ML for Diabetes Risk Prediction

A reproducible machine learning pipeline for diabetes risk classification on the CDC Diabetes Health
Indicators dataset (BRFSS 2015), comparing Logistic Regression, Random Forest, XGBoost, and LightGBM,
with particular emphasis on probability calibration and SHAP-based interpretability, which are less
commonly evaluated alongside discrimination and subgroup analysis in comparable diabetes-risk studies.

**Full source, all figures, and version history:** https://github.com/manasvi-sahare/diabetes-prediction

All outputs shown below are real, captured from an actual run of this pipeline — nothing here is
illustrative or simulated.


## Step 0 — Setup: Install Packages

This notebook runs fully self-contained in the Colab session's local storage (no Google Drive permissions needed). Trade-off: nothing persists between sessions, so every fresh runtime re-downloads the dataset and retrains all models from scratch (Random Forest's 5-fold CV step is the slowest part, taking several minutes).

In [1]:
import os
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('figures', exist_ok=True)
print('Working directory:', os.getcwd())

Working directory: /content


In [1]:
!pip install xgboost lightgbm shap --quiet
import xgboost, lightgbm, shap, sklearn, pandas, numpy
print('xgboost:', xgboost.__version__)
print('lightgbm:', lightgbm.__version__)
print('shap:', shap.__version__)
print('scikit-learn:', sklearn.__version__)


xgboost: 2.1.1
lightgbm: 4.5.0
shap: 0.46.0
scikit-learn: 1.5.2


### Download the dataset

Re-downloads on every fresh session (no persistent storage in this version).

In [1]:
import os

data_path = 'data/diabetes.csv'
!curl -sL -o {data_path} "https://raw.githubusercontent.com/Helmy2/Diabetes-Health-Indicators/main/diabetes_binary_health_indicators_BRFSS2015.csv"
print("Downloaded fresh copy.")

import pandas as pd
print("File size:", os.path.getsize(data_path) / 1e6, "MB")

Downloaded fresh copy.
File size: 22.7 MB


## Step 1 — Data Exploration

CDC Diabetes Health Indicators dataset, derived from the 2015 BRFSS (Behavioral Risk Factor Surveillance System), UCI ML Repository, DOI: 10.24432/C53919.

In [1]:
df = pd.read_csv('data/diabetes.csv')

print("Shape:", df.shape)
print("\nTarget distribution:")
print(df['Diabetes_binary'].value_counts())
print(f"\nClass imbalance ratio: {df['Diabetes_binary'].value_counts()[0] / df['Diabetes_binary'].value_counts()[1]:.2f} : 1")
print(f"Positive (diabetic) rate: {df['Diabetes_binary'].mean():.2%}")
print(f"\nMissing values: {df.isnull().sum().sum()}")


Shape: (253680, 22)

Target distribution:
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64

Class imbalance ratio: 6.18 : 1
Positive (diabetic) rate: 13.93%

Missing values: 0


## Step 2 — Feature Types

Classifying each of the 21 features by cardinality to understand the mix of binary, ordinal, and continuous predictors.

In [1]:
for col in df.columns:
    nunique = df[col].nunique()
    vmin, vmax = df[col].min(), df[col].max()
    if nunique == 2:
        ftype = "binary flag"
    elif nunique <= 6:
        ftype = "ordinal (small scale)"
    elif nunique <= 14:
        ftype = "ordinal (larger scale)"
    else:
        ftype = "continuous"
    print(f"{col:<22}{nunique:<10}{vmin:<8}{vmax:<8}{ftype}")


Diabetes_binary       2         0.0     1.0     binary flag
HighBP                2         0.0     1.0     binary flag
HighChol              2         0.0     1.0     binary flag
CholCheck             2         0.0     1.0     binary flag
BMI                   84        12.0    98.0    continuous
Smoker                2         0.0     1.0     binary flag
Stroke                2         0.0     1.0     binary flag
HeartDiseaseorAttack  2         0.0     1.0     binary flag
PhysActivity          2         0.0     1.0     binary flag
Fruits                2         0.0     1.0     binary flag
Veggies               2         0.0     1.0     binary flag
HvyAlcoholConsump     2         0.0     1.0     binary flag
AnyHealthcare         2         0.0     1.0     binary flag
NoDocbcCost           2         0.0     1.0     binary flag
GenHlth               5         1.0     5.0     ordinal (small scale)
MentHlth              31        0.0     30.0    continuous
PhysHlth              31        

## Step 3 — Train/Test Split & Scaling

Stratified 80/20 split preserves the true 13.93% positive rate in both sets.

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train set: {X_train.shape}, positive rate: {y_train.mean():.2%}")
print(f"Test set:  {X_test.shape}, positive rate: {y_test.mean():.2%}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"\nScaled mean (first 3 features): {X_train_scaled[:, :3].mean(axis=0).round(4)}")
print(f"Scaled std  (first 3 features): {X_train_scaled[:, :3].std(axis=0).round(4)}")

joblib.dump((X_train, X_test, y_train, y_test, scaler), 'models/split.pkl')


Train set: (202944, 21), positive rate: 13.93%
Test set:  (50736, 21), positive rate: 13.93%

Scaled mean (first 3 features): [-0. -0.  0.]
Scaled std  (first 3 features): [1. 1. 1.]


## Step 4 — Model: Logistic Regression (Interpretable Baseline)

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                              average_precision_score, brier_score_loss, confusion_matrix)

logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train_scaled, y_train)

proba = logreg.predict_proba(X_test_scaled)[:, 1]
pred = (proba >= 0.5).astype(int)

print(f"Precision: {precision_score(y_test, pred):.4f}")
print(f"Recall:    {recall_score(y_test, pred):.4f}")
print(f"F1:        {f1_score(y_test, pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, proba):.4f}")
print(f"PR-AUC:    {average_precision_score(y_test, proba):.4f}")
print(f"Brier:     {brier_score_loss(y_test, proba):.4f}")

joblib.dump(logreg, 'models/logreg.pkl')


Precision: 0.3107
Recall:    0.7611
F1:        0.4413
ROC-AUC:   0.8196
PR-AUC:    0.3926
Brier:     0.1776


## Step 5 — Model: Random Forest (Regularized)

**Note:** an early unregularized version of this model (`max_depth=None`) produced a 441MB model file with no performance benefit. `max_depth=12, min_samples_leaf=20` cuts the file to 9.2MB with statistically indistinguishable performance — a real engineering lesson from this project.

In [1]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=20,
                             max_features='sqrt', class_weight='balanced', n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

proba = rf.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(f"Precision: {precision_score(y_test, pred):.4f}")
print(f"Recall:    {recall_score(y_test, pred):.4f}")
print(f"F1:        {f1_score(y_test, pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, proba):.4f}")
print(f"PR-AUC:    {average_precision_score(y_test, proba):.4f}")
print(f"Brier:     {brier_score_loss(y_test, proba):.4f}")

joblib.dump(rf, 'models/random_forest.pkl')
import os
print(f"\nModel file size: {os.path.getsize('models/random_forest.pkl')/1e6:.1f} MB")


Precision: 0.3083
Recall:    0.7772
F1:        0.4415
ROC-AUC:   0.8238
PR-AUC:    0.4177
Brier:     0.1738

Model file size: 9.2 MB


## Step 6 — Model: XGBoost (Imbalance-Corrected)

In [1]:
import xgboost as xgb

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight = {scale_pos_weight:.3f}")

xgb_model = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                               scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                               n_jobs=-1, random_state=42)
xgb_model.fit(X_train, y_train)

proba = xgb_model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(f"\nPrecision: {precision_score(y_test, pred):.4f}")
print(f"Recall:    {recall_score(y_test, pred):.4f}")
print(f"F1:        {f1_score(y_test, pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, proba):.4f}")
print(f"PR-AUC:    {average_precision_score(y_test, proba):.4f}")
print(f"Brier:     {brier_score_loss(y_test, proba):.4f}")

joblib.dump(xgb_model, 'models/xgboost.pkl')


scale_pos_weight = 6.177

Precision: 0.3093
Recall:    0.7752
F1:        0.4422
ROC-AUC:   0.8236
PR-AUC:    0.4173
Brier:     0.1714


## Step 7 — Model: LightGBM

In [1]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                       scale_pos_weight=scale_pos_weight, n_jobs=-1, random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)

proba = lgbm.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(f"Precision: {precision_score(y_test, pred):.4f}")
print(f"Recall:    {recall_score(y_test, pred):.4f}")
print(f"F1:        {f1_score(y_test, pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, proba):.4f}")
print(f"PR-AUC:    {average_precision_score(y_test, proba):.4f}")
print(f"Brier:     {brier_score_loss(y_test, proba):.4f}")

joblib.dump(lgbm, 'models/lgbm.pkl')
import os
print(f"\nModel file size: {os.path.getsize('models/lgbm.pkl')/1e6:.1f} MB")


Precision: 0.3070
Recall:    0.7860
F1:        0.4415
ROC-AUC:   0.8254
PR-AUC:    0.4194
Brier:     0.1734

Model file size: 1.0 MB


## Step 8 — Probability Calibration

Comparing sigmoid (Platt) vs. isotonic calibration on XGBoost. This is the core novel contribution: testing whether predicted probabilities can be trusted at face value, not just whether the model ranks patients correctly.

In [1]:
from sklearn.calibration import CalibratedClassifierCV

base = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                          scale_pos_weight=scale_pos_weight, eval_metric='logloss',
                          n_jobs=-1, random_state=42)

cal_sigmoid = CalibratedClassifierCV(base, method='sigmoid', cv=3)
cal_sigmoid.fit(X_train, y_train)
proba_sig = cal_sigmoid.predict_proba(X_test)[:, 1]

cal_isotonic = CalibratedClassifierCV(base, method='isotonic', cv=3)
cal_isotonic.fit(X_train, y_train)
proba_iso = cal_isotonic.predict_proba(X_test)[:, 1]

proba_raw = xgb_model.predict_proba(X_test)[:, 1]

print("=== XGBoost + sigmoid calibration ===")
print(f"AUC-ROC: {roc_auc_score(y_test, proba_sig):.4f}   Brier: {brier_score_loss(y_test, proba_sig):.4f}")
print("\n=== XGBoost + isotonic calibration ===")
print(f"AUC-ROC: {roc_auc_score(y_test, proba_iso):.4f}   Brier: {brier_score_loss(y_test, proba_iso):.4f}")
print("\n=== XGBoost (uncalibrated, original) ===")
print(f"AUC-ROC: {roc_auc_score(y_test, proba_raw):.4f}   Brier: {brier_score_loss(y_test, proba_raw):.4f}")

# Isotonic wins (narrowly) -- save as the final calibrated model
joblib.dump(cal_isotonic, 'models/xgboost_calibrated.pkl')
print("\nSaved best-calibrated model (isotonic) to models/xgboost_calibrated.pkl")


=== XGBoost + sigmoid calibration ===
AUC-ROC: 0.8235   Brier: 0.0981

=== XGBoost + isotonic calibration ===
AUC-ROC: 0.8232   Brier: 0.0981

=== XGBoost (uncalibrated, original) ===
AUC-ROC: 0.8236   Brier: 0.1714

Saved best-calibrated model (isotonic) to models/xgboost_calibrated.pkl


**Key finding:** Brier Score improves 43% (0.1714 -> 0.0981) with no significant loss in ROC-AUC. Raw XGBoost is badly over-confident when trained with imbalance correction -- e.g. at a predicted probability of ~0.62, the true observed rate is only ~0.21. Calibration corrects this.

## Step 9 — Threshold Tuning

Finding the F1-optimal decision threshold on the calibrated model, rather than blindly using the default 0.5.

In [1]:
import numpy as np
from sklearn.metrics import precision_recall_curve

proba_cal = cal_isotonic.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, proba_cal)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Best threshold by F1: {best_threshold:.3f}")
print(f"  Precision: {precision[best_idx]:.3f}")
print(f"  Recall:    {recall[best_idx]:.3f}")
print(f"  F1:        {f1_scores[best_idx]:.3f}")

target_recall = 0.75
idx_recall = np.argmin(np.abs(recall[:-1] - target_recall))
print(f"\nThreshold for ~{target_recall:.0%} recall: {thresholds[idx_recall]:.3f}")
print(f"  Precision at that threshold: {precision[idx_recall]:.3f}")

TUNED_THRESHOLD = 0.245  # locked in from this analysis, reused throughout the rest of the notebook


Best threshold by F1: 0.245
  Precision: 0.374
  Recall:    0.616
  F1:        0.465

Threshold for ~75% recall: 0.160
  Precision at that threshold: 0.320


## Step 10 — SHAP Explainability (Global)

In [1]:
import shap

explainer = shap.TreeExplainer(xgb_model)
sample_idx = np.random.RandomState(42).choice(X_test.index, size=3000, replace=False)
X_sample = X_test.loc[sample_idx]
shap_values = explainer(X_sample)

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_importance = pd.DataFrame({
    'feature': X_test.columns, 'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print("Top 10 features by SHAP importance:")
print(shap_importance.head(10).to_string(index=False))
shap_importance.to_csv('figures/shap_importance.csv', index=False)

shap.summary_plot(shap_values, X_sample, max_display=12)


Top 10 features by SHAP importance:
             feature  mean_abs_shap
             GenHlth       0.663138
              HighBP       0.557153
                 Age       0.436933
                 BMI       0.430112
            HighChol       0.313866
              Income       0.146377
                 Sex       0.139515
            MentHlth       0.084814
           CholCheck       0.081858
HeartDiseaseorAttack       0.073157


## Step 11 — SHAP Local Explanation (Single Patient)

Explaining one individual prediction, not just population-wide patterns.

In [1]:
diabetic_idx = y_test[y_test == 1].index[0]
patient = X_test.loc[[diabetic_idx]]
shap_val = explainer(patient)
proba_patient = xgb_model.predict_proba(patient)[0, 1]

print(f"Patient index: {diabetic_idx}")
print(f"Actual label: {'Diabetic' if y_test.loc[diabetic_idx] == 1 else 'Non-diabetic'}")
print(f"Predicted probability: {proba_patient:.3f}")

shap.plots.waterfall(shap_val[0], max_display=10)


Patient index: 225051
Actual label: Diabetic
Predicted probability: 0.774


## Step 12 — Final Results Summary (All Base Models + Calibration + Tuned Threshold)

In [1]:
results = []
for name, proba_arr, thresh in [
    ('Logistic Regression', logreg.predict_proba(X_test_scaled)[:, 1], 0.5),
    ('Random Forest', rf.predict_proba(X_test)[:, 1], 0.5),
    ('XGBoost', proba_raw, 0.5),
    ('XGBoost (calibrated, thresh=0.5)', proba_cal, 0.5),
    ('XGBoost (calibrated, tuned thresh=0.245)', proba_cal, 0.245),
    ('LightGBM', lgbm.predict_proba(X_test)[:, 1], 0.5),
]:
    pred = (proba_arr >= thresh).astype(int)
    results.append({
        'Model': name, 'Threshold': thresh,
        'Precision': round(precision_score(y_test, pred), 4),
        'Recall': round(recall_score(y_test, pred), 4),
        'F1': round(f1_score(y_test, pred), 4),
        'ROC_AUC': round(roc_auc_score(y_test, proba_arr), 4),
        'PR_AUC': round(average_precision_score(y_test, proba_arr), 4),
        'Brier': round(brier_score_loss(y_test, proba_arr), 4),
    })

summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))
summary_df.to_csv('figures/final_results_summary.csv', index=False)


                                    Model  Threshold  Precision  Recall     F1  ROC_AUC  PR_AUC   Brier
                     Logistic Regression      0.500     0.3107  0.7611 0.4413   0.8196  0.3926  0.1776
                           Random Forest      0.500     0.3083  0.7772 0.4415   0.8238  0.4177  0.1738
                                 XGBoost      0.500     0.3093  0.7752 0.4422   0.8236  0.4173  0.1714
        XGBoost (calibrated, thresh=0.5)      0.500     0.5624  0.1338 0.2162   0.8232  0.4091  0.0981
XGBoost (calibrated, tuned thresh=0.245)      0.245     0.3798  0.6002 0.4652   0.8232  0.4091  0.0981
                                LightGBM      0.500     0.3070  0.7860 0.4415   0.8254  0.4194  0.1734


## Step 13 — Statistical Significance Testing

1,000-resample paired bootstrap on the test set: are the differences between models real, or just noise?

In [1]:
from sklearn.metrics import roc_auc_score, brier_score_loss

N_BOOTSTRAP = 1000
rng = np.random.RandomState(42)
y_test_arr = y_test.values
n = len(y_test_arr)

proba_by_model = {
    'Logistic Regression': logreg.predict_proba(X_test_scaled)[:, 1],
    'Random Forest': rf.predict_proba(X_test)[:, 1],
    'XGBoost': proba_raw,
    'XGBoost (calibrated)': proba_cal,
    'LightGBM': lgbm.predict_proba(X_test)[:, 1],
}

boot_indices = [rng.choice(n, size=n, replace=True) for _ in range(N_BOOTSTRAP)]
auc_boot_by_model = {}

for name, proba in proba_by_model.items():
    aucs = []
    for idx in boot_indices:
        y_b, p_b = y_test_arr[idx], proba[idx]
        if len(np.unique(y_b)) < 2:
            continue
        aucs.append(roc_auc_score(y_b, p_b))
    auc_boot_by_model[name] = np.array(aucs)
    auc_ci = np.percentile(aucs, [2.5, 97.5])
    print(f"{name}: AUC={roc_auc_score(y_test_arr, proba):.4f}  95% CI=[{auc_ci[0]:.4f}, {auc_ci[1]:.4f}]")

print("\n=== Pairwise AUC Difference (paired bootstrap) ===")
pairs = [('Logistic Regression', 'Random Forest'), ('Logistic Regression', 'XGBoost'),
         ('Random Forest', 'XGBoost'), ('XGBoost', 'XGBoost (calibrated)'),
         ('XGBoost', 'LightGBM'), ('Random Forest', 'LightGBM')]
for a, b in pairs:
    diff = auc_boot_by_model[a] - auc_boot_by_model[b]
    ci = np.percentile(diff, [2.5, 97.5])
    sig = not (ci[0] <= 0 <= ci[1])
    point_diff = roc_auc_score(y_test_arr, proba_by_model[a]) - roc_auc_score(y_test_arr, proba_by_model[b])
    print(f"{a} vs {b}: diff={point_diff:+.4f}, 95% CI=[{ci[0]:+.4f}, {ci[1]:+.4f}], "
          f"{'SIGNIFICANT' if sig else 'not significant'}")


Logistic Regression: AUC=0.8196  95% CI=[0.8147, 0.8240]
Random Forest: AUC=0.8238  95% CI=[0.8186, 0.8282]
XGBoost: AUC=0.8236  95% CI=[0.8187, 0.8280]
XGBoost (calibrated): AUC=0.8232  95% CI=[0.8184, 0.8277]
LightGBM: AUC=0.8254  95% CI=[0.8206, 0.8299]

=== Pairwise AUC Difference (paired bootstrap) ===
Logistic Regression vs Random Forest: diff=-0.0042, 95% CI=[-0.0056, -0.0025], SIGNIFICANT
Logistic Regression vs XGBoost: diff=-0.0040, 95% CI=[-0.0058, -0.0019], SIGNIFICANT
Random Forest vs XGBoost: diff=+0.0002, 95% CI=[-0.0013, +0.0017], not significant
XGBoost vs XGBoost (calibrated): diff=+0.0004, 95% CI=[-0.0006, +0.0014], not significant
XGBoost vs LightGBM: diff=-0.0018, 95% CI=[-0.0026, -0.0009], SIGNIFICANT
Random Forest vs LightGBM: diff=-0.0016, 95% CI=[-0.0028, -0.0004], SIGNIFICANT


**Interpretation:** Ensemble methods (RF, XGBoost, LightGBM) all significantly outperform the
linear baseline. Random Forest and XGBoost are statistically indistinguishable from each other.
LightGBM has a small but statistically significant edge (~0.002 AUC) over both -- a useful reminder
that **statistical significance is not the same as practical significance**: with a 50,736-case test
set, the bootstrap is sensitive enough to detect differences far below any threshold that would
change a real-world screening decision.

## Step 14 — Independent 5-Fold Cross-Validation

Confirming the bootstrap findings with an entirely independent method: stratified 5-fold CV on the training set.

In [1]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for model_name in ['Logistic Regression', 'Random Forest', 'XGBoost']:
    aucs = []
    for tr_idx, val_idx in skf.split(X_train, y_train):
        Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        ytr, yval = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        if model_name == 'Logistic Regression':
            fs = StandardScaler()
            Xtr_s, Xval_s = fs.fit_transform(Xtr), fs.transform(Xval)
            m = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
            m.fit(Xtr_s, ytr)
            auc = roc_auc_score(yval, m.predict_proba(Xval_s)[:, 1])
        elif model_name == 'Random Forest':
            m = RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=20,
                                        max_features='sqrt', class_weight='balanced', n_jobs=-1, random_state=42)
            m.fit(Xtr, ytr)
            auc = roc_auc_score(yval, m.predict_proba(Xval)[:, 1])
        else:  # XGBoost
            spw = (ytr == 0).sum() / (ytr == 1).sum()
            m = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                   scale_pos_weight=spw, eval_metric='logloss', n_jobs=-1, random_state=42)
            m.fit(Xtr, ytr)
            auc = roc_auc_score(yval, m.predict_proba(Xval)[:, 1])
        aucs.append(auc)

    cv_results.append({'Model': model_name, 'CV_AUC_mean': np.mean(aucs), 'CV_AUC_std': np.std(aucs)})
    print(f"{model_name}: fold AUCs = {[round(a,4) for a in aucs]}")
    print(f"  Mean: {np.mean(aucs):.4f} +/- {np.std(aucs):.4f}\n")

cv_summary = pd.DataFrame(cv_results).round(4)
print(cv_summary.to_string(index=False))
cv_summary.to_csv('figures/cv_all_models_summary.csv', index=False)


Logistic Regression: fold AUCs = [0.8209, 0.825, 0.8245, 0.8233, 0.8228]
  Mean: 0.8233 +/- 0.0014

Random Forest: fold AUCs = [0.8246, 0.8281, 0.83, 0.8306, 0.8279]
  Mean: 0.8282 +/- 0.0021

XGBoost: fold AUCs = [0.823, 0.8277, 0.8278, 0.8287, 0.8244]
  Mean: 0.8263 +/- 0.0022

              Model  CV_AUC_mean  CV_AUC_std
Logistic Regression       0.8233      0.0014
      Random Forest       0.8282      0.0021
            XGBoost       0.8263      0.0022


## Step 15 — Subgroup Performance & Calibration Analysis

*Note: this analysis reports subgroup performance and calibration differences, not a formal fairness
audit -- it does not test against defined fairness criteria (e.g. demographic parity, equalized odds).*

None of the six papers reviewed in this project's literature review tested whether performance and
calibration hold up across demographic subgroups -- so this pipeline includes that check.

In [1]:
def subgroup_metrics(mask, group_name, group_label, proba_arr, threshold=0.245):
    y_g = y_test[mask].values
    p_g = proba_arr[mask.values]
    pred_g = (p_g >= threshold).astype(int)
    if len(np.unique(y_g)) < 2:
        return None
    return {
        'Group': group_name, 'Subgroup': group_label, 'N': int(mask.sum()),
        'Prevalence': round(y_g.mean(), 4),
        'ROC_AUC': round(roc_auc_score(y_g, p_g), 4),
        'Brier': round(brier_score_loss(y_g, p_g), 4),
        'Recall': round(recall_score(y_g, pred_g, zero_division=0), 4),
    }

results = []
for val, label in [(0, 'Female'), (1, 'Male')]:
    r = subgroup_metrics(X_test['Sex'] == val, 'Sex', label, proba_cal)
    if r: results.append(r)

age_buckets = {'18-39': X_test['Age'].between(1, 4), '40-59': X_test['Age'].between(5, 8),
               '60+': X_test['Age'].between(9, 13)}
for label, mask in age_buckets.items():
    r = subgroup_metrics(mask, 'Age', label, proba_cal)
    if r: results.append(r)

income_buckets = {'Lower (1-4)': X_test['Income'].between(1, 4), 'Higher (5-8)': X_test['Income'].between(5, 8)}
for label, mask in income_buckets.items():
    r = subgroup_metrics(mask, 'Income', label, proba_cal)
    if r: results.append(r)

subgroup_df = pd.DataFrame(results)
print(subgroup_df.to_string(index=False))
subgroup_df.to_csv('figures/subgroup_performance.csv', index=False)


Group     Subgroup     N  Prevalence  ROC_AUC  Brier  Recall
  Sex       Female 28531      0.1292   0.8334 0.0912  0.5943
  Sex         Male 22205      0.1524   0.8093 0.1069  0.6067
  Age        18-39  7749      0.0333   0.8367 0.0282  0.1938
  Age        40-59 18361      0.1061   0.8316 0.0783  0.5280
  Age          60+ 24626      0.1974   0.7698 0.1348  0.6508
Income  Lower (1-4) 11418      0.2268   0.7812 0.1455  0.7305
Income Higher (5-8) 39318      0.1139   0.8244 0.0843  0.5249


**Key subgroup findings:**
- ROC-AUC drops from 0.837 (ages 18-39) to 0.770 (ages 60+) -- discrimination genuinely degrades with age.
- Recall for ages 18-39 is only 0.194 at the globally-tuned threshold -- not a discrimination failure
  (this group has the *best* AUC of any age bucket) but a threshold-calibration failure, since this
  subgroup's prevalence (3.3%) is far below the population average (13.9%) the threshold was tuned against.
- Lower-income individuals have nearly double the diabetes prevalence of higher-income individuals
  (22.7% vs. 11.4%) but the model discriminates worse for them (AUC 0.781 vs. 0.824).

### Expected Calibration Error (ECE) by Subgroup

Brier Score conflates *discrimination difficulty* with *calibration quality*. ECE isolates calibration specifically by measuring the gap between predicted and observed probability within bins.

In [1]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_prob, bin_edges[1:-1])
    ece, n = 0.0, len(y_true)
    for b in range(n_bins):
        mask = bin_indices == b
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(y_prob[mask].mean() - y_true[mask].mean())
    return ece

print(f"{'Group':<20}{'Brier':<10}{'ECE':<10}")
for label, mask in {**age_buckets, **income_buckets}.items():
    y_g, p_g = y_test.values[mask.values], proba_cal[mask.values]
    brier = brier_score_loss(y_g, p_g)
    ece = expected_calibration_error(y_g, p_g)
    print(f"{label:<20}{brier:<10.4f}{ece:<10.4f}")


Group               Brier     ECE       
18-39               0.0282    0.0036    
40-59               0.0783    0.0054    
60+                 0.1348    0.0055    
Lower (1-4)         0.1455    0.0160    
Higher (5-8)        0.0843    0.0033    


**Important correction to the Brier-based story:** ECE reveals that Age's Brier gap (0.0783 -> 0.1348)
is mostly a *discrimination-difficulty* artifact (harder-to-classify predictions cluster near the
decision boundary, inflating Brier even under good calibration), NOT a true calibration failure --
ECE is nearly identical for 40-59 and 60+ (0.0054 vs 0.0055). **Income, however, shows a genuine
calibration failure**: ECE for lower-income individuals is ~5x worse (0.0160 vs 0.0033) -- a real,
distinct problem from the discrimination gap.

## Step 16 — SHAP Stability Across CV Folds

The research question: are SHAP explanations stable, or do the 'important features' change depending on the training split? We retrain each model across the same 5 CV folds and compute SHAP rankings per fold.

In [1]:
from scipy.stats import spearmanr

def shap_stability(model_name, fit_and_explain_fn):
    fold_importances, fold_top10, fold_top5 = [], [], []
    for tr_idx, val_idx in skf.split(X_train, y_train):
        Xtr, Xval = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        ytr = y_train.iloc[tr_idx]
        mean_abs_shap = fit_and_explain_fn(Xtr, ytr, Xval)
        importance = pd.Series(mean_abs_shap, index=X_train.columns).sort_values(ascending=False)
        fold_importances.append(importance)
        fold_top10.append(set(importance.head(10).index))
        fold_top5.append(set(importance.head(5).index))

    importance_df = pd.DataFrame(fold_importances).T
    rank_df = importance_df.rank(ascending=False)
    correlations = [spearmanr(rank_df.iloc[:, i], rank_df.iloc[:, j])[0]
                     for i in range(5) for j in range(i + 1, 5)]

    def mean_jaccard(sets):
        js = [len(sets[i] & sets[j]) / len(sets[i] | sets[j]) for i in range(len(sets)) for j in range(i+1, len(sets))]
        return np.mean(js)

    return {'Model': model_name, 'mean_spearman_rho': round(np.mean(correlations), 4),
            'top5_jaccard': round(mean_jaccard(fold_top5), 4), 'top10_jaccard': round(mean_jaccard(fold_top10), 4)}


def xgb_explain(Xtr, ytr, Xval):
    spw = (ytr == 0).sum() / (ytr == 1).sum()
    m = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, scale_pos_weight=spw,
                           eval_metric='logloss', n_jobs=-1, random_state=42)
    m.fit(Xtr, ytr)
    idx = np.random.RandomState(42).choice(Xval.index, size=min(3000, len(Xval)), replace=False)
    return np.abs(shap.TreeExplainer(m)(Xval.loc[idx]).values).mean(axis=0)

def lr_explain(Xtr, ytr, Xval):
    fs = StandardScaler()
    Xtr_s, Xval_s = fs.fit_transform(Xtr), fs.transform(Xval)
    m = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    m.fit(Xtr_s, ytr)
    idx = np.random.RandomState(42).choice(len(Xval_s), size=min(3000, len(Xval_s)), replace=False)
    return np.abs(shap.LinearExplainer(m, Xtr_s)(Xval_s[idx]).values).mean(axis=0)

def rf_explain(Xtr, ytr, Xval):
    m = RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=20,
                                max_features='sqrt', class_weight='balanced', n_jobs=-1, random_state=42)
    m.fit(Xtr, ytr)
    idx = np.random.RandomState(42).choice(Xval.index, size=min(3000, len(Xval)), replace=False)
    vals = shap.TreeExplainer(m)(Xval.loc[idx]).values
    if vals.ndim == 3:
        vals = vals[:, :, 1]
    return np.abs(vals).mean(axis=0)

def lgbm_explain(Xtr, ytr, Xval):
    spw = (ytr == 0).sum() / (ytr == 1).sum()
    m = LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, scale_pos_weight=spw,
                        n_jobs=-1, random_state=42, verbose=-1)
    m.fit(Xtr, ytr)
    idx = np.random.RandomState(42).choice(Xval.index, size=min(3000, len(Xval)), replace=False)
    return np.abs(shap.TreeExplainer(m)(Xval.loc[idx]).values).mean(axis=0)

stability_results = [
    shap_stability('Logistic Regression', lr_explain),
    shap_stability('Random Forest', rf_explain),
    shap_stability('XGBoost', xgb_explain),
    shap_stability('LightGBM', lgbm_explain),
]
stability_df = pd.DataFrame(stability_results)
print(stability_df.to_string(index=False))
stability_df.to_csv('figures/shap_stability_all_models.csv', index=False)


               Model  mean_spearman_rho  top5_jaccard  top10_jaccard
Logistic Regression             0.9853        1.0000         0.9273
       Random Forest             0.9957        1.0000         1.0000
             XGBoost             0.9948        1.0000         0.8909
            LightGBM             0.9905        1.0000         1.0000


**Finding:** SHAP rankings are highly stable across resampled training data for every model tested
(Spearman rho > 0.98 in all cases), and every model agrees perfectly on the top-5 features across
every fold (Jaccard = 1.0). This holds regardless of whether the model is linear (Logistic Regression)
or tree-based (RF/XGBoost/LightGBM) -- the explanations are not an artifact of one particular training
split.

## Step 17 — Threshold Sensitivity Sweep

Is the F1-optimal threshold (0.245) a sharp, unique optimum, or part of a broad, forgiving plateau?

In [1]:
thresholds_sweep = np.arange(0.10, 0.91, 0.05)
sweep_results = []
for t in thresholds_sweep:
    pred = (proba_cal >= t).astype(int)
    sweep_results.append({
        'Threshold': round(t, 2),
        'Precision': round(precision_score(y_test, pred, zero_division=0), 4),
        'Recall': round(recall_score(y_test, pred, zero_division=0), 4),
        'F1': round(f1_score(y_test, pred, zero_division=0), 4),
        'Positive_Pred_Rate': round(pred.mean(), 4),
    })
sweep_df = pd.DataFrame(sweep_results)
print(sweep_df.to_string(index=False))

best_row = sweep_df.loc[sweep_df['F1'].idxmax()]
near_optimal = sweep_df[sweep_df['F1'] >= best_row['F1'] * 0.98]
print(f"\nF1-optimal (coarse sweep): threshold={best_row['Threshold']}, F1={best_row['F1']}")
print(f"Near-optimal plateau (within 2% of best F1): {near_optimal['Threshold'].min()} to {near_optimal['Threshold'].max()}")
sweep_df.to_csv('figures/threshold_sensitivity.csv', index=False)


Threshold  Precision  Recall     F1  Positive_Pred_Rate
     0.10     0.2658  0.8706 0.4073              0.4563
     0.15     0.3070  0.7814 0.4408              0.3547
     0.20     0.3489  0.6697 0.4588              0.2674
     0.25     0.3798  0.6002 0.4652              0.2202
     0.30     0.4327  0.4514 0.4419              0.1453
     0.35     0.4663  0.3614 0.4072              0.1080
     0.40     0.4982  0.2543 0.3368              0.0711
     0.45     0.5252  0.1942 0.2836              0.0515
     0.50     0.5624  0.1338 0.2162              0.0332
     0.55     0.5873  0.1103 0.1858              0.0262
     0.60     0.6216  0.0546 0.1004              0.0122
     0.65     0.6438  0.0279 0.0534              0.0060
     0.70     0.6520  0.0252 0.0485              0.0054
     0.75     0.7069  0.0116 0.0228              0.0023
     0.80     0.7059  0.0068 0.0135              0.0013
     0.85     0.6800  0.0024 0.0048              0.0005
     0.90     0.0000  0.0000 0.0000             

**Finding:** the F1 peak is real and narrow (0.20-0.25), not a broad plateau -- confirming the
earlier fine-grained result (0.245) as a genuine local optimum. But even within this narrow window, a
real precision/recall trade-off exists: threshold 0.20 catches more true positives (recall 0.670) at
the cost of more false alarms, while 0.25 is more conservative (precision 0.380). Positive Prediction
Rate translates this into operational terms -- ranging from 22.0% to 26.7% of the population flagged
across this narrow band alone.

## Step 18 — Subgroup Gap Across All Models

Is the age-related discrimination gap specific to one algorithm, or a property of the data itself?

In [1]:
gap_results = []
for name, proba_arr in [
    ('Logistic Regression', logreg.predict_proba(X_test_scaled)[:, 1]),
    ('Random Forest', rf.predict_proba(X_test)[:, 1]),
    ('XGBoost', proba_raw),
    ('LightGBM', lgbm.predict_proba(X_test)[:, 1]),
]:
    subgroup_aucs = {}
    for label, mask in age_buckets.items():
        y_g, p_g = y_test.values[mask.values], proba_arr[mask.values]
        if len(np.unique(y_g)) < 2:
            continue
        subgroup_aucs[label] = roc_auc_score(y_g, p_g)
    gap = max(subgroup_aucs.values()) - min(subgroup_aucs.values())
    row = {'Model': name, 'Subgroup_Gap_AgeAUC': round(gap, 4)}
    row.update({f'AUC_{k}': round(v, 4) for k, v in subgroup_aucs.items()})
    gap_results.append(row)
    print(f"{name}: gap={gap:.4f}  " + '  '.join(f'{k}={v:.4f}' for k, v in subgroup_aucs.items()))

gap_df = pd.DataFrame(gap_results)
gap_df.to_csv('figures/subgroup_gap_all_models.csv', index=False)


Logistic Regression: gap=0.0736  18-39=0.8368  40-59=0.8325  60+=0.7632
Random Forest: gap=0.0728  18-39=0.8435  40-59=0.8324  60+=0.7707
XGBoost: gap=0.0668  18-39=0.8377  40-59=0.8318  60+=0.7709
LightGBM: gap=0.0725  18-39=0.8454  40-59=0.8330  60+=0.7729


**Finding:** all four models show the same pattern -- discrimination for ages 60+ lands in the
same 0.76-0.77 range regardless of algorithm, while 18-39 and 40-59 both sit around 0.83-0.85. The
gap-across-models spread is narrow (0.0668 to 0.0736). This is model-agnostic: the age-related
performance gap is inherent to the signal available in the features for older adults, not an
artifact of any one algorithm's assumptions.

## Step 19 — Master Results Table

Combining every axis of evaluation performed in this notebook: discrimination, calibration, interpretability stability, and subgroup behavior.

In [1]:
master_table = pd.DataFrame([
    {'Model': 'Logistic Regression', 'ROC_AUC': 0.8196, 'PR_AUC': 0.3926, 'F1': 0.4413,
     'Brier_raw': 0.1776, 'Brier_calibrated': None, 'SHAP_Stability_rho': 0.9853, 'Subgroup_Gap_AgeAUC': 0.0736},
    {'Model': 'Random Forest', 'ROC_AUC': 0.8238, 'PR_AUC': 0.4177, 'F1': 0.4415,
     'Brier_raw': 0.1738, 'Brier_calibrated': None, 'SHAP_Stability_rho': 0.9957, 'Subgroup_Gap_AgeAUC': 0.0728},
    {'Model': 'XGBoost', 'ROC_AUC': 0.8236, 'PR_AUC': 0.4173, 'F1': 0.4422,
     'Brier_raw': 0.1714, 'Brier_calibrated': 0.0981, 'SHAP_Stability_rho': 0.9948, 'Subgroup_Gap_AgeAUC': 0.0668},
    {'Model': 'LightGBM', 'ROC_AUC': 0.8254, 'PR_AUC': 0.4194, 'F1': 0.4415,
     'Brier_raw': 0.1734, 'Brier_calibrated': None, 'SHAP_Stability_rho': 0.9905, 'Subgroup_Gap_AgeAUC': 0.0725},
])
print(master_table.to_string(index=False))
master_table.to_csv('figures/master_results_table.csv', index=False)

print('''
Note: Brier_calibrated is only populated for XGBoost, since that is the only model this pipeline
wrapped in CalibratedClassifierCV. SHAP_Stability_rho and Subgroup_Gap_AgeAUC were extended to all
four models in Steps 16 and 18 above.
''')


               Model  ROC_AUC  PR_AUC     F1  Brier_raw  Brier_calibrated  SHAP_Stability_rho  Subgroup_Gap_AgeAUC
Logistic Regression   0.8196  0.3926 0.4413     0.1776               NaN              0.9853               0.0736
      Random Forest   0.8238  0.4177 0.4415     0.1738               NaN              0.9957               0.0728
            XGBoost   0.8236  0.4173 0.4422     0.1714            0.0981              0.9948               0.0668
            Subgroup_Gap_AgeAUC
            LightGBM   0.8254  0.4194 0.4415     0.1734               NaN              0.9905               0.0725


## Research Questions & Findings

| # | Research Question | Finding |
|---|---|---|
| RQ1 | Do nonlinear ensemble models significantly outperform logistic regression? | **Yes** -- RF/XGBoost/LightGBM all beat LR by a statistically significant margin (bootstrap 95% CI excludes 0) |
| RQ2 | Does probability calibration improve reliability without degrading discrimination? | **Yes** -- Brier Score improves 43% with no significant AUC change |
| RQ3 | Does threshold optimization improve F1 under class imbalance? | **Yes** -- F1 improves from 0.216 (thresh=0.5) to 0.465 (thresh=0.245), though this involves a real precision/recall trade-off |
| RQ4 | Are model performance and calibration consistent across demographic subgroups? | **No** -- real, quantified disparities by Age (discrimination) and Income (calibration, via ECE) |
| RQ5 | Are observed model-vs-model differences statistically significant AND practically meaningful? | **Mixed** -- LightGBM's edge over RF/XGBoost is statistically significant but practically negligible (~0.002 AUC) |

## Limitations

- **Self-reported, cross-sectional survey data** (BRFSS) -- not lab-confirmed diagnoses; introduces recall/social-desirability bias
- **No external validation cohort** -- all results come from a held-out split of the same dataset, strengthened by bootstrap CI and independent 5-fold CV, but not tested on a separate data source
- **Subgroup analysis is not a formal fairness audit** -- no defined fairness criteria (demographic parity, equalized odds) were tested
- **Prediction, not diagnosis** -- this model estimates statistical risk from survey features; it does not replace clinical assessment

## Conclusion

This notebook systematically evaluates discrimination, calibration, interpretability stability,
threshold selection, statistical significance, and subgroup behavior for diabetes-risk prediction
models -- not just "which model gets the highest accuracy." The most actionable findings are: (1)
algorithm choice matters far less than calibration and threshold tuning, (2) a class-imbalance-corrected
model can be badly overconfident even with good discrimination, and (3) subgroup performance gaps are
real, quantifiable, and consistent across model architectures -- meaning they reflect the data and
task, not a fixable modeling artifact.

**Full source, all figures, and version history:** https://github.com/manasvi-sahare/diabetes-prediction
